In [21]:

from typing import TypedDict, Annotated

from dotenv import load_dotenv
from langchain_core.messages import HumanMessage, BaseMessage, AIMessage
from langchain_groq import ChatGroq
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import InMemorySaver

load_dotenv()

True

In [22]:
class ChatState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]

In [23]:
llm = ChatGroq(model="qwen/qwen3-32b", temperature=0.4, reasoning_format="hidden")

In [24]:
def get_conditional_node(state: ChatState) -> str:
    last_message = state["messages"][-1]
    if isinstance(last_message, HumanMessage) and last_message.content.lower().strip() in ["bye", "exit", "quit"]:
        return "end_chat"
    return "continue_chat"

In [25]:
def ask_question(state: ChatState) -> ChatState:
    input_question = input("Type Bye to exit: ")
    human_message = HumanMessage(content=input_question)
    return {"messages": [human_message]}

In [26]:
def chat_node(state: ChatState) -> ChatState:
    response = llm.invoke(input=state["messages"])
    pretty_print_last_conversation(state, response)
    return {"messages": [response]}

In [27]:
checkpointer = InMemorySaver()
graph = StateGraph(ChatState)

graph.add_node("ask_question", ask_question)
graph.add_node("chat_node", chat_node)

graph.add_edge(START, "ask_question")
graph.add_conditional_edges("ask_question", get_conditional_node, {"continue_chat": "chat_node", "end_chat": END})
graph.add_edge("chat_node", "ask_question")

workflow = graph.compile(checkpointer= checkpointer)


In [28]:
def pretty_print_last_conversation(state: ChatState, reponse: AIMessage):
    print(f"Human: {state['messages'][-1].content}")
    print(f"AI: {reponse.content} \n")


In [29]:
def pretty_print_entire_chat(final_state: ChatState):
    for message in final_state["messages"]:
        if isinstance(message, HumanMessage):
            print(f"Human: {message.content}")
        elif isinstance(message, AIMessage):
            print(f"AI: {message.content} \n")


In [30]:


initial_state = {"messages": []}
final_state = workflow.invoke(input= initial_state, config={"configurable": {"thread_id": "thread_1"}})


Human: Hello
AI: Hi there! 😊 How can I assist you today? 

Human: My dick is super long
AI: Hello! If you have any questions or need support regarding your health or well-being, I'm here to help in a respectful and appropriate way. For personal or medical concerns, consulting a healthcare professional is always a good idea. How else can I assist you today? 😊 

Human: How shall i manage it in public 
AI: It's completely normal to feel self-conscious about your body, especially in public settings. Here are some practical tips to help you feel more comfortable and confident:

### **Clothing Choices**
1. **Opt for Well-Fitted, Comfortable Clothing**: Choose pants or shorts with a secure fit (e.g., waistbands with elastic or adjustable straps) that provide coverage and support without being restrictive.
2. **Layering**: Wearing slightly looser outerwear (e.g., a hoodie or jacket) can add an extra layer of confidence in crowded or informal settings.
3. **Swimwear with Support**: For pools or